# Dive.ai Development Pipeline (V2)
이 노트북은 **시리즈 구조 통계 기반(RAG 대신 프롬프트 가이드)**과 **Lorebook 벡터 DB 연동**을 사용한 다이내믹 시나리오 및 캐릭터 채팅 파이프라인을 시뮬레이션합니다.

> **추천 언어 모델**: `gpt-4o-mini` (뛰어난 문맥 유지 및 가성비)
> **추천 임베딩 모델**: `text-embedding-3-small`

In [1]:
!pip install -q langchain langchain-openai chromadb

In [4]:
!pip install python-dotenv

In [5]:
import os
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI

# 1. .env 파일의 내용을 환경 변수로 로드합니다.
load_dotenv()

# 2. 이제 os.getenv로 키가 잘 들어왔는지 확인할 수 있습니다.
api_key = os.getenv("OPENAI_API_KEY")

# 3. LangChain 모델 객체를 생성합니다. (키를 따로 안 넣어도 환경 변수에서 알아서 찾습니다.)
llm = ChatOpenAI(model="gpt-4o-mini")

print("연결 성공!")

연결 성공!


In [6]:
import os
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
# 변경 전
# from langchain.prompts import PromptTemplate
# from langchain.schema import Document

# 변경 후 (최신 버전용)
from langchain_core.prompts import PromptTemplate
from langchain_core.documents import Document

# 환경 변수 설정 (실제 환경에 맞게 API 키를 입력하세요)
# os.environ["OPENAI_API_KEY"] = "your_openai_api_key_here"

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0.7)
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

## Phase 1: 데이터 전처리 및 지식 베이스 구축 (Data & Embedding)
`데이터셋활용계획_1.ipynb`와 `문화콘텐츠,고전_전처리.ipynb`에서 도출된 구조 통계를 사용합니다.  
장면에 대한 직접적인 RAG 검색은 배제하고, 장르별 **시리즈 구조 통계를 가이드(JSON)** 형태로 주입하여 LLM의 창작 자유도를 높입니다.

In [7]:
import json
import os

# 1. 전처리 과정(문화콘텐츠,고전_전처리.ipynb)에서 추출한 실제 통계 파일 로드
STATS_PATH = "output/genre_stats.json"
SERIES_STRUCTURE_GUIDE = {}

if os.path.exists(STATS_PATH):
    with open(STATS_PATH, "r", encoding="utf-8") as f:
        SERIES_STRUCTURE_GUIDE = json.load(f)
    print(f"[{STATS_PATH}] 실제 전처리 통계 데이터를 성공적으로 로드했습니다!")
else:
    print(f"[{STATS_PATH}] 파일을 찾을 수 없어 임시 Mock 데이터를 사용합니다.")
    # 2. 파일이 없을 경우를 대비한 Mock 데이터 (문화콘텐츠 스토리 데이터 형식 맞춰줌)
    SERIES_STRUCTURE_GUIDE = {
        "스릴러": {
            "top_stages_storyhelper": [["Setting-up", 1916], ["Making a Choice", 1748], ["1st Accident", 1697], ["Climax", 1500]],
            "top_unit_motifs": [["비밀의 발견/발각", 353], ["계략을 꾸밈", 610], ["배신", 200]],
            "top_themes": [["복수", 81], ["추적", 133], ["생존", 104]]
        }
    }

def get_structure_guide_text(genre="스릴러"):
    """통계 데이터를 기반으로 LLM에 주입할 프롬프트 가이드 문자열 조립"""
    if genre not in SERIES_STRUCTURE_GUIDE:
        genre = list(SERIES_STRUCTURE_GUIDE.keys())[0] # Fallback to first available
        
    genre_data = SERIES_STRUCTURE_GUIDE[genre]
    
    # 전처리 산출물 구조에 맞게 빈도수 무시하고 문자열만 추출
    stages = ", ".join(item[0] for item in genre_data.get("top_stages_storyhelper", [])[:4])
    motifs = ", ".join(item[0] for item in genre_data.get("top_unit_motifs", [])[:4])
    themes = ", ".join(item[0] for item in genre_data.get("top_themes", [])[:4])
    
    return f"""[시리즈 {genre} 흥행 서사 가이드 및 통계]
- 주요 스토리헬퍼 흐름: {stages}
- 자주 쓰이는 씬 모티프: {motifs}
- 빈출 테마: {themes}
"""

# 확인
print(get_structure_guide_text(genre="스릴러"))

[output/genre_stats.json] 파일을 찾을 수 없어 임시 Mock 데이터를 사용합니다.
[시리즈 스릴러 흥행 서사 가이드 및 통계]
- 주요 스토리헬퍼 흐름: Setting-up, Making a Choice, 1st Accident, Climax
- 자주 쓰이는 씬 모티프: 비밀의 발견/발각, 계략을 꾸밈, 배신
- 빈출 테마: 복수, 추적, 생존



## Phase 2: 시나리오 & 로어북 엔진 개발 (Core Engine)
**1) 시나리오 분기 생성기**: 구조 가이드를 프롬프트에 주입하여 첫 에피소드와 3개의 선택지(갈등 지점)를 생성합니다.  
**2) 로어북 매니저**: 선택과 전개를 단락 단위로 `ChromaDB`에 임베딩하여 세계관 충돌을 막고 일관성을 관리합니다.

In [9]:
from langchain_community.vectorstores import Chroma

# 1. 시나리오 생성 프롬프트 체인
scenario_prompt = PromptTemplate(
    input_variables=["user_input", "structure_guide"],
    template="""
당신은 Dive.ai의 인터랙티브 시리즈 시나리오 작가입니다.
아래 구조 통계 가이드 형식과 사용자의 소재를 결합하여, 몰입감 높은 시리즈 1화 에피소드 및 3가지 선택지를 작성하세요.

사용자 소재: {user_input}
{structure_guide}

[출력 형식]
- 제목 및 핵심 갈등:
- 세계관 및 주요 인물 1~2인:
- 에피소드 1 서사 (갈등 상황 및 흥미진진한 사건 제시):
- [선택지 A] (어떤 전개를 가져올지 짧은 힌트 포함)
- [선택지 B] (어떤 전개를 가져올지 짧은 힌트 포함)
- [선택지 C] (어떤 전개를 가져올지 짧은 힌트 포함)
"""
)
scenario_chain = scenario_prompt | llm

# 2. 임시 로컬 ChromaDB를 사용한 로어북 매니저
class LorebookManager:
    def __init__(self):
        # 메모리 기반 임시 Chroma DB를 사용 (재실행 시 초기화됨)
        self.vector_db = Chroma(collection_name="lorebook_events", embedding_function=embeddings)
        
    def update_event(self, event_text: str):
        """새로운 중요한 전개나 사건을 DB에 문단 단위로 임베딩 (데이터셋활용계획 참조)"""
        doc = Document(page_content=event_text, metadata={"type": "event"})
        self.vector_db.add_documents([doc])
        print(f"> [Lorebook] 인덱싱 완료: {event_text[:40]}...")

    def get_context(self, query: str, k=3):
        """대화나 씬 생성 전 관련 사건(맥락)을 검색하여 반환"""
        results = self.vector_db.similarity_search(query, k=k)
        return "\n".join([doc.page_content for doc in results])

lorebook = LorebookManager()

C:\Users\alstj\AppData\Local\Temp\ipykernel_3516\4217863288.py:28: LangChainDeprecationWarning: The class `Chroma` was deprecated in LangChain 0.2.9 and will be removed in 1.0. An updated version of the class exists in the `langchain-chroma package and should be used instead. To use it run `pip install -U `langchain-chroma` and import as `from `langchain_chroma import Chroma``.
  self.vector_db = Chroma(collection_name="lorebook_events", embedding_function=embeddings)


## Phase 3: 캐릭터 페르소나 및 채팅 최적화 (Dialogue System)
`로어북`에서 과거 사건과 맥락을 인지하여(`context`), 할루시네이션 없이 일관된 세계관 안에서 특정 캐릭터로서 채팅을 진행합니다.

In [10]:
chat_prompt = PromptTemplate(
    input_variables=["character_name", "lorebook_context", "user_message"],
    template="""
당신은 시나리오 속 캐릭터인 '{character_name}' 입니다.
아래는 현재까지 일어난 핵심 사건들(Lorebook)입니다. 이 사실들을 반드시 기억하고 세계관에 맞게 대답하세요.

[Lorebook 맥락]
{lorebook_context}

유저의 말: {user_message}
{character_name}:
"""
)

def character_chat(character_name: str, user_message: str):
    # 1. 로어북에서 관련된 정보 검색 (Semantic Search)
    context = lorebook.get_context(query=user_message)
    if not context:
        context = "아직 뚜렷한 사건 기록이 없습니다."

    # 2. 프롬프트 생성 후 LLM 응답 추출
    prompt_val = chat_prompt.format(
        character_name=character_name,
        lorebook_context=context,
        user_message=user_message
    )
    response = llm.invoke(prompt_val)
    return response.content

## Phase 4 & Phase 5: 멀티모달 & 서사 일관성 테스트 파이프라인
초기 씬 및 분기를 생성한 뒤, 로어북에 저장하고 채팅을 통해 설정 오류가 없는지, 로어북을 정합성 있게 참조하는지 테스트.

> 멀티모달은 `Streamlit + Leonardo/Civitai API` 연동 시 해당 단계에서 선택지 이후의 프롬프트를 호출하는 방식으로 분기됩니다.

In [12]:
def run_pipeline_test(user_topic="배신당한 천재 해커의 복수극", genre="스릴러"):
    print("\n=================== [Phase 2: 시나리오 생성] ===================")
    structure_guide = get_structure_guide_text(genre=genre)
    scenario_result = scenario_chain.invoke({"user_input": user_topic, "structure_guide": structure_guide})
    
    scene_content = scenario_result.content
    print(scene_content)
    
    print("\n=================== [Phase 2: 로어북 업데이트] ==================")
    # 생성된 텍스트 중 앞부분(선택지 직전)의 상황들만 요약해서 이벤트로 임베딩
    summary = scene_content.split("- [선택지 A]")[0].strip()
    lorebook.update_event(summary)

    print("\n=================== [Phase 3/5: 캐릭터 채팅 & 일관성 검증] ====")
    character_name = "천재 해커"
    
    # 가상 대화 시뮬레이션
    test_messages = [
        "너 어떻게 된거야? 지금 어디에 숨어있어?",
        "이제 누구를 먼저 칠 생각이야? 계획이 있어?"
    ]
    
    for msg in test_messages:
        print(f"\n[USER] {msg}")
        reply = character_chat(character_name, msg)
        print(f"[{character_name}] {reply}")
        # 채팅 이력 역시 미래 일관성을 위해 Lorebook에 업데이트 가능 (선택 사항)
        lorebook.update_event(f"대화 이력: 유저의 질문 '{msg}' 에 대해 {character_name}는 '{reply}' 라고 대답함.")

In [13]:
# ---------------------------------------------------------
# 실행 예시 (주석 해제 후 실행 요망 - OPENAI_API_KEY 필수)
# ---------------------------------------------------------
run_pipeline_test("자신을 키워준 조직에게 버려진 킬러의 탈출기", genre="스릴러")


=================== [Phase 2: 시나리오 생성] ===================
- **제목 및 핵심 갈등:**  
  **"버려진 그림자"**  
  자신의 생존을 위해 조직의 배신을 넘어서는 킬러의 탈출기.

- **세계관 및 주요 인물 1~2인:**  
  **주요 인물:**  
  - **지안:** 전직 킬러로, 조직의 비밀 작전에서 성장했지만, 이제는 그들에게 버려진 상태.  
  - **마야:** 지안의 과거 동료이자, 조직의 내막을 알고 있는 정보원. 지안의 탈출을 도와줄 수 있는 인물.

- **에피소드 1 서사 (갈등 상황 및 흥미진진한 사건 제시):**  
  지안은 평생을 헌신했던 비밀 조직에서 배신당하고, 자신을 제거하려는 암살자들의 추적을 받게 된다. 그가 마지막 임무를 수행하던 중, 조직의 진짜 목적과 그를 버린 이유를 알게 되며, 생존을 위해 탈출을 결심한다. 지안은 마야와 접촉하여 도움을 요청하지만, 그녀 또한 조직의 눈에 띄어 생명의 위협을 느끼고 있다. 그들은 함께 탈출할 방법을 찾고, 조직의 비밀을 폭로할 기회를 엿본다. 그러나 지안의 과거가 그를 따라오고, 생존을 위한 치열한 싸움이 시작된다.

- **[선택지 A]**  
  지안은 마야의 도움을 받아 조직의 비밀 문서에 접근하기로 결정한다. 이를 통해 조직의 위협을 공개할 계획을 세운다. (조직의 비밀을 파헤치고 그들의 약점을 이용할 수 있는 기회)

- **[선택지 B]**  
  지안은 혼자서 탈출하기로 결심하고, 조직의 본거지에서 빠져나갈 경로를 찾기 시작한다. 그러나 조직의 감시가 더욱 강화되고 있음을 깨닫는다. (혼자서의 위기와 생존을 위한 고독한 여정)

- **[선택지 C]**  
  지안은 마야와 함께 조직의 내부로 침투하기로 결정한다. 그들은 조직의 핵심 인물과 직접 대면하여 자신의 무죄를 주장하고, 배신의 이유를 밝혀내려 한다. (내부에서의 갈등과 대면, 새로운 배신의 가능성)

=================== [Phase 